# Model? 

Weights + Forward Pass

In [ ]:
import torch

## Tensor?

In [ ]:
# A single number
scalar = torch.tensor(3.14)
print(f"Scalar: {scalar}")
print(f"  Shape: {scalar.shape}")  # no dimensions

# A list of numbers (1D tensor = vector)
vector = torch.tensor([1.0, 2.0, 3.0, 4.0])
print(f"\nVector: {vector}")
print(f"  Shape: {vector.shape}")  # 4 numbers

# A grid of numbers (2D tensor = matrix)
matrix = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
print(f"\nMatrix:\n{matrix}")
print(f"  Shape: {matrix.shape}")  # 3 rows, 2 columns


In [ ]:
# ============================================================
# PART 2: A model's "weights" are just tensors of random numbers
#         that get adjusted during training
# ============================================================

# Imagine a tiny model with just 2 weights
weights = torch.tensor([0.5, -0.3])
bias = torch.tensor([0.1])

print("\n--- Simplest possible model ---")
print(f"Weights: {weights}")
print(f"Bias:    {bias}")

# The "forward pass" = multiply input by weights, add bias
input_data = torch.tensor([2.0, 3.0])
output = torch.dot(weights, input_data) + bias  # (0.5*2) + (-0.3*3) + 0.1 = 0.2


In [ ]:
# ============================================================
# PART 3: A neural network = layers of these operations
# ============================================================

# Let's build a tiny neural network with PyTorch's nn module
import torch.nn as nn

class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Layer 1: takes 4 inputs, produces 8 outputs
        self.layer1 = nn.Linear(4, 8)
        # Layer 2: takes 8 inputs, produces 2 outputs
        self.layer2 = nn.Linear(8, 2)
        # Activation: adds non-linearity (lets the model learn curves, not just lines)
        self.activation = nn.ReLU()

    def forward(self, x):
        x = self.layer1(x)       # multiply by weights + bias
        x = self.activation(x)   # zero out negatives
        x = self.layer2(x)       # multiply by weights + bias again
        return x

model = TinyModel()

print("\n--- A tiny neural network ---")
print(f"Model:\n{model}")

In [ ]:
# Let's count the weights
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters (weights): {total_params}")

# Let's SEE the weights — they're just random numbers right now
print(f"\nLayer 1 weights shape: {model.layer1.weight.shape}")
print(f"Layer 1 weights (random!):\n{model.layer1.weight.data}")

# Run a forward pass — feed in 4 numbers, get 2 numbers out
input_data = torch.tensor([1.0, 2.0, 3.0, 4.0])
output = model(input_data)
print(f"\nInput:  {input_data}")
print(f"Output: {output}")
print("(The output is meaningless right now — the weights are random!)")


In [ ]:

print("\n--- Scaling up ---")
print(f"Our tiny model:      {total_params:>15,} parameters")
print(f"Llama 3.2 1B:        {1_000_000_000:>15,} parameters")
print(f"Llama 3.2 3B:        {3_000_000_000:>15,} parameters")
print(f"Llama 3.1 70B:       {70_000_000_000:>15,} parameters")

print(f"\nMemory (FP16 = 2 bytes per parameter):")
print(f"Our tiny model:      {total_params * 2:>15,} bytes")
print(f"Llama 3.2 3B:        {3_000_000_000 * 2:>15,} bytes = ~6 GB")
print(f"Llama 3.1 70B:       {70_000_000_000 * 2:>15,} bytes = ~140 GB")

# ============================================================
# KEY TAKEAWAY:
#
# A model = weights (random numbers) + forward pass (multiply recipe)
# Training = adjusting those random numbers until the outputs are useful
#
# An LLM like Llama 3.2 3B is this EXACT same concept,
# just with 3 billion weights instead of 42.
#
# Next: we'll learn how training adjusts those weights.
# ============================================================


# Training

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class Adder(nn.Module): 
    """ A simple model that adds two numbers together. """
    def __init__(self) :
        super().__init__()

        self.layer1 = nn.Linear(2,16)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(16,1)
    def forward(self, x):
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        return x

model = Adder()
print(model)

In [ ]:
print("=== BEFORE TRAINING (random weights) ===")
test_cases = [[3.0, 5.0], [10.0, 20.0], [1.0, 1.0]]
for a, b in test_cases:
    inp = torch.tensor([a, b])
    pred = model(inp).item()
    print(f"  {a} + {b} = {pred:.4f}  (should be {a + b})")


## Training - The 4 Step Loop

1. Forward
2. Loss
3. Backward
4. Update

In [ ]:
# MSE = mean squared error
loss_fn = nn.MSELoss()

# Optimizer = the thing that adjusts the weights based on the loss 
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


for step in range(2000) :
    a = torch.rand(1)*20 
    b = torch.rand(1)*20

    input_data = torch.stack([a,b], dim=1).squeeze()

    correct_answer = (a + b).squeeze()


    # Forward pass
    pred = model(input_data).squeeze()

    # Compute loss
    loss = loss_fn(pred, correct_answer)

    # Backward Pass
    optimizer.zero_grad()  # clear previous gradients
    loss.backward()        # compute new gradients

    # Update weights
    optimizer.step()       # adjust weights based on gradients

    if step % 200 == 0 :
        print(f"Step {step:>4}: Loss = {loss.item():.4f}")

In [ ]:
# ============================================================
# PART 3: Test the trained model — it should know addition now!
# ============================================================

print("\n=== AFTER TRAINING ===")
for a, b in test_cases:
    inp = torch.tensor([a, b])
    pred = model(inp).item()
    correct = a + b
    error = abs(pred - correct)
    print(f"  {a} + {b} = {pred:.4f}  (should be {correct}, error: {error:.4f})")


In [ ]:
# Test on numbers it has NEVER seen
print("\n=== GENERALIZATION (never seen these during training) ===")
new_cases = [[7.5, 2.5], [15.0, 15.0], [0.0, 0.0], [19.9, 0.1]]
for a, b in new_cases:
    inp = torch.tensor([a, b])
    pred = model(inp).item()
    correct = a + b
    error = abs(pred - correct)
    print(f"  {a} + {b} = {pred:.4f}  (should be {correct}, error: {error:.4f})")


# A Real Tiny Language Model

We try out a LM on our poor 4GB GPU..

In [ ]:
# these libraries do the following:
# - AutoModelForCausalLM: loads a pre-trained language model that can generate text
# - AutoTokenizer: loads the tokenizer that converts text to numbers and back
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "gpt2"  # a small, pre-trained language model from Hugging Face

# Define tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


In [ ]:
text = "The cat sat on the mat."

tokens = tokenizer.encode(text)

print(f"Original text: {text}")
print(f"Tokens: {tokens}")
print(f"Decoded text: {tokenizer.decode(tokens)}")

print("Token breakdown")

for token_id in tokens : 
    token_text = tokenizer.decode([token_id])
    print(f"  Token ID: {token_id}  --> '{token_text}'")
print("Vocalbulary size:", tokenizer.vocab_size)

In [ ]:
math_text = "Question: What is 16-3 -4? Answer: 16-3 = 13, 13 - 4 = 9"
math_tokens = tokenizer.encode(math_text)

print(f"\nMath text: {math_text}")
print(f"Tokens: {math_tokens}")
print(f"Decoded text: {tokenizer.decode(math_tokens)}")

# breakdown 
print("\nToken breakdown for math text:")
for token_id in math_tokens :
    token_text = tokenizer.decode([token_id])
    print(f"  Token ID: {token_id}  --> '{token_text}'")


In [ ]:
# Give the model a prompt — it predicts what comes next
prompt = "The capital of France is" 
input_ids = tokenizer.encode(prompt, return_tensors="pt").to("cuda")

# Forward pass: model scores every possible next token
with torch.no_grad():
    outputs = model(input_ids)
    next_token_logits = outputs.logits[0, -1, :]  # scores for position after last token

# Convert scores to probabilities
probs = torch.softmax(next_token_logits, dim=0)

# Top 10 most likely next tokens
top10 = torch.topk(probs, 10)
print(f"Prompt: '{prompt}'\n")
print(f"Top 10 predicted next tokens:")
for i in range(10):
    token_id = top10.indices[i].item()
    prob = top10.values[i].item()
    token_text = tokenizer.decode([token_id])
    marker = " ★" if i == 0 else ""
    print(f"  {prob:>6.2%}  '{token_text}'{marker}")


In [ ]:
prompt = "2 + 2 ="

input_ids = tokenizer.encode(prompt, return_tensors="pt").to("cuda")
generated_ids = input_ids.clone()

print(f"Starting prompt: '{prompt}'")
print("-"*50)

for step in range(15) :
    with torch.no_grad():
        outputs = model(generated_ids)
        probs = torch.softmax(outputs.logits[0, -1, :], dim=0)
        next_token_id = torch.argmax(probs) # pick the most likely next token

    generated_ids = torch.cat([generated_ids, next_token_id.unsqueeze(0).unsqueeze(0)], dim=1)
    new_token = tokenizer.decode([next_token_id])
    full_text = tokenizer.decode(generated_ids[0])

    print(f"  Step {step+1:>2d}: picked '{new_token}' ({probs[next_token_id]:.1%}) → '{full_text}'")

In [ ]:
import plotly.graph_objects as go

# Before training
test_prompt = "Question: What is 3 + 4? Answer:"
input_ids = tokenizer.encode(test_prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    output_ids = model.generate(input_ids, max_new_tokens=5, do_sample=False)
before = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(f"BEFORE TRAINING:")
print(f"  '{test_prompt}' → '{before}'\n")

# Training data
training_data = [
    "Question: What is 2 + 3? Answer: 5",
    "Question: What is 4 + 1? Answer: 5",
    "Question: What is 7 + 2? Answer: 9",
    "Question: What is 3 + 3? Answer: 6",
    "Question: What is 5 + 5? Answer: 10",
    "Question: What is 8 + 1? Answer: 9",
    "Question: What is 6 + 3? Answer: 9",
    "Question: What is 1 + 1? Answer: 2",
    "Question: What is 9 + 1? Answer: 10",
    "Question: What is 4 + 4? Answer: 8",
]

# Training with loss tracking
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
model.train()

epoch_losses = []
print("TRAINING...")
for epoch in range(30):
    total_loss = 0
    for text in training_data:
        tokens = tokenizer.encode(text, return_tensors="pt").to("cuda")
        outputs = model(tokens, labels=tokens)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(training_data)
    epoch_losses.append(avg_loss)
    if epoch % 5 == 0:
        print(f"  Epoch {epoch:>2d} | Loss: {avg_loss:.4f}")

model.eval()
print("Done!\n")

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(len(epoch_losses))),
    y=epoch_losses,
    mode='lines+markers',
    name='Training Loss',
    line=dict(color='#636EFA', width=2),
    marker=dict(size=4),
))
fig.update_layout(
    title="SFT Training Loss — GPT-2 Learning Addition",
    xaxis_title="Epoch",
    yaxis_title="Loss (lower = better)",
    template="plotly_dark",
    height=400,
)
fig.show()


In [ ]:
# Test on both seen and unseen problems
test_prompts = [
    ("2 + 3", "5",  True),   # seen in training
    ("5 + 5", "10", True),   # seen
    ("1 + 1", "2",  True),   # seen
    ("3 + 4", "7",  False),  # UNSEEN
    ("6 + 6", "12", False),  # UNSEEN
    ("2 + 7", "9",  False),  # UNSEEN
    ("8 + 2", "10", False),  # UNSEEN
]

results = []
print("AFTER TRAINING:\n")
for question, correct, seen in test_prompts:
    prompt = f"Question: What is {question}? Answer:"
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = model.generate(input_ids, max_new_tokens=5, do_sample=False)
    full = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    answer = full[len(prompt):].strip().split()[0] if full[len(prompt):].strip() else "?"

    is_correct = answer == correct
    tag = "SEEN" if seen else "NEW "
    mark = "✓" if is_correct else "✗"
    print(f"  [{tag}] {question} = {answer:>4s}  (correct: {correct})  {mark}")
    results.append({"question": question, "answer": answer, "correct": correct,
                     "is_correct": is_correct, "seen": seen})

# Visualize
seen_acc = sum(1 for r in results if r["seen"] and r["is_correct"]) / sum(1 for r in results if r["seen"])
new_acc = sum(1 for r in results if not r["seen"] and r["is_correct"]) / max(1, sum(1 for r in results if not r["seen"]))

fig = go.Figure(data=[
    go.Bar(name="Seen in training", x=["Accuracy"], y=[seen_acc * 100], marker_color="#636EFA"),
    go.Bar(name="Unseen (generalization)", x=["Accuracy"], y=[new_acc * 100], marker_color="#EF553B"),
])
fig.update_layout(
    title="Does GPT-2 Generalize? Seen vs Unseen Math Problems",
    yaxis_title="Accuracy %",
    yaxis_range=[0, 105],
    template="plotly_dark",
    barmode="group",
    height=400,
)
fig.show()


# LoRA 

Freese all original weights, add tiny trainable adapters.

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to("cuda")

# pad_token is needed for batching inputs of different lengths
# eos means "end of sequence" — we can use that as a padding token since it won't appear in the middle of our inputs

# Padding is necessary when we want to do batching, which is when we feed multiple inputs into the model at once for efficiency.
tokenizer.pad_token = tokenizer.eos_token

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters in the model: {total_params}")

# Freeze everything
for param  in model.parameters():
    param.requires_grad = False

trainable_before = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("After freezing all layers, trainable parameters:", trainable_before)

# Note: Model is now READ ONLY. LoRA adds trainable adapters on top of this frozen model.

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Total parameters in the model: 124439808
After freezing all layers, trainable parameters: 0


In [3]:
from peft import LoraConfig, get_peft_model

# LoRA config: which layers to adapt and how small the adapters should be
lora_config = LoraConfig(
    r= 8, # Rank - how many numbers per adapter
    lora_alpha=16, # Scaling factor for the adapters
    target_modules=["c_attn", "c_proj"], # which layers to add adapters to (the attention layers)
    lora_dropout=0.1, # regularization to prevent overfitting
)

# Wrap the frozen model with LoRA adapters 
lora_model = get_peft_model(model, lora_config)


# Count parameters now
total = sum(p.numel() for p in lora_model.parameters())
trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
frozen = total - trainable

print("=== WITH LoRA ===")
print(f"Total parameters:     {total:>12,}")
print(f"Frozen (original):    {frozen:>12,}  ← untouched")
print(f"Trainable (adapters): {trainable:>12,}  ← only these update")
print(f"Trainable %:          {trainable/total*100:>11.2f}%")
print(f"\nLoRA added just {trainable:,} parameters on top of {frozen:,}")
print(f"That's {frozen // trainable}x fewer parameters to train!")

=== WITH LoRA ===
Total parameters:      125,250,816
Frozen (original):     124,439,808  ← untouched
Trainable (adapters):      811,008  ← only these update
Trainable %:                 0.65%

LoRA added just 811,008 parameters on top of 124,439,808
That's 153x fewer parameters to train!


/home/manas/anaconda3/envs/math-reasoning-distill/lib/python3.10/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/manas/anaconda3/envs/math-reasoning-distill/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [4]:
from datasets import load_dataset
import plotly.graph_objects as go

# Load the REAL dataset we'll use in the project
gsm8k = load_dataset("openai/gsm8k","main")

print("Sample problem from GSM8K:")
print(gsm8k["train"][0]["question"])

# lenth of test and train
print(f"\nTraining set size: {len(gsm8k['train']):,} problems")
print(f"Test set size:     {len(gsm8k['test']):,} problems")

# let's look at some examples

for i in range(3) :
    example = gsm8k["train"][i]
    print(f"\nExample {i+1}:")
    print("Question:", example["question"])
    print("Answer:", example["answer"])
    print("-----")

Sample problem from GSM8K:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

Training set size: 7,473 problems
Test set size:     1,319 problems

Example 1:
Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Answer: Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72
-----

Example 2:
Question: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?
Answer: Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.
#### 10
-----

Example 3:
Question: Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for

In [5]:
# Format examples into prompt-completion pairs .
def format_example(example) :
    return f"Question: {example['question']} \n Answer: {example['answer']}"

# Use 50 examples (GPT-2 is tiny, this enough to see learning)
train_data = [format_example(gsm8k["train"][i]) for i in range(50)]

# Print the first 3 formatted examples to verify
print("\nFormatted training examples:")

print(f"Training on {len(train_data)} examples:")


# # Train : LoRA 
# optimizer = torch.optim.AdamW(
#     filter(lambda p: p.requires_grad, lora_model.parameters()), 
#     lr=5e-4
# )

# lora_model.train()

# lora_losses = []

# print("Training with LoRA on GSM8K")

# for epoch in range(20):
#     total_loss = 0
#     for text in train_data : 
#         tokens = tokenizer.encode(text, return_tensors="pt", truncation=True, max_length=512).to("cuda")
#         outputs = lora_model(tokens, labels=tokens)
#         loss = outputs.loss

#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()
#         total_loss += loss.item()
#     avg_loss = total_loss / len(train_data)
#     lora_losses.append(avg_loss)
#     if epoch % 5 == 0:
#         print(f"  Epoch {epoch:>2d} | Loss: {avg_loss:.4f}")
# # Eval
# lora_model.eval()
# print("Done training LoRA adapters!")


# # Plot 
# fig = go.Figure()
# fig.add_trace(go.Scatter(
#     x=list(range(len(lora_losses))),
#     y=lora_losses,
#     mode='lines+markers',
#     name='LoRA Training Loss',
#     line=dict(color='#EF553B', width=2),
#     marker=dict(size=4),
# ))
# fig.update_layout(
#     title="LoRA Training Loss on GSM8K",
#     xaxis_title="Epoch",
#     yaxis_title="Loss (lower = better)",
#     template="plotly_dark",
#     height=400,
# )
# fig.show()


Formatted training examples:
Training on 50 examples:


In [6]:
%load_ext tensorboard
from torch.utils.tensorboard import SummaryWriter

# Launch tensorboard inline in the notebook
%tensorboard --logdir runs/lora_gsm8k

In [7]:
writer = SummaryWriter("runs/lora_gsm8k")

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, lora_model.parameters()),
    lr=5e-4
)
lora_model.train()

global_step = 0
for epoch in range(20):
    total_loss = 0
    for text in train_data:
        tokens = tokenizer.encode(text, return_tensors="pt").to("cuda")
        outputs = lora_model(tokens, labels=tokens)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        global_step += 1
        writer.add_scalar("loss/step", loss.item(), global_step)

    avg_loss = total_loss / len(train_data)
    writer.add_scalar("loss/epoch", avg_loss, epoch)
    if epoch % 4 == 0:
        print(f"  Epoch {epoch:>2d} | Loss: {avg_loss:.4f}")

writer.close()
lora_model.eval()
print("Done!")

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


  Epoch  0 | Loss: 2.8596
  Epoch  4 | Loss: 1.8574
  Epoch  8 | Loss: 1.5573
  Epoch 12 | Loss: 1.2797
  Epoch 16 | Loss: 1.0843
Done!


In [8]:
# Test problems — mix of seen (from training 50) and completely new
test_cases = [
    # From training set (first 50)
    {"q": gsm8k["train"][0]["question"], "a": gsm8k["train"][0]["answer"], "seen": True},
    {"q": gsm8k["train"][5]["question"], "a": gsm8k["train"][5]["answer"], "seen": True},
    # Never seen during training
    {"q": gsm8k["train"][100]["question"], "a": gsm8k["train"][100]["answer"], "seen": False},
    {"q": gsm8k["train"][200]["question"], "a": gsm8k["train"][200]["answer"], "seen": False},
    {"q": gsm8k["train"][500]["question"], "a": gsm8k["train"][500]["answer"], "seen": False},
]

# Extract ground truth answer (number after ####)
def extract_answer(answer_text):
    if "####" in answer_text:
        return answer_text.split("####")[-1].strip()
    return "?"

# Generate with LoRA model
print("=== LoRA MODEL ON GSM8K ===\n")
for i, tc in enumerate(test_cases):
    tag = "SEEN" if tc["seen"] else "NEW "
    correct = extract_answer(tc["a"])
    
    prompt = f"Question: {tc['q']}\nAnswer:"
    input_ids = tokenizer.encode(prompt, return_tensors="pt", truncation=True, max_length=400).to("cuda")
    
    with torch.no_grad():
        output_ids = lora_model.generate(
            input_ids, max_new_tokens=150, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated = tokenizer.decode(output_ids[0][input_ids.shape[1]:], skip_special_tokens=True)
    model_answer = extract_answer(generated) if "####" in generated else generated[:80]
    
    print(f"[{tag}] Problem {i+1}")
    print(f"  Q: {tc['q'][:80]}...")
    print(f"  Correct answer: {correct}")
    print(f"  Model output: {model_answer}")
    print()


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


=== LoRA MODEL ON GSM8K ===

[SEEN] Problem 1
  Q: Natalia sold clips to 48 of her friends in April, and then she sold half as many...
  Correct answer: 72
  Model output: 72 clips altogether

[SEEN] Problem 2
  Q: Mark has a garden with flowers. He planted plants of three different colors in i...
  Correct answer: 35
  Model output: 85.
The number of pink flowers is 85% * 10 = <<85%*10=85>>85.
The number of yellow flowers is 85% * 15 = <<85*15=85>>

[NEW ] Problem 3
  Q: A craft store makes a third of its sales in the fabric section, a quarter of its...
  Correct answer: 15
  Model output:  The number of sales in the fabric section was 36 + 24 = <<36+24=36>>36.
The jew

[NEW ] Problem 4
  Q: Sansa is a famous artist, she can draw a portrait and sell it according to its s...
  Correct answer: 195
  Model output: 3500/day = $<<3500/day=3500>>3500 per day

[NEW ] Problem 5
  Q: Joe played catch with Derek and Tammy. He caught the ball 23 times. Derek made f...
  Correct answer: 30
  Mode

# GRPO

In [9]:
"""
GRPO with code verification:

  1. Model sees: "What is 16 - 3 - 4?"
  2. Model generates 4 code solutions:
  
     Solution A:                Solution B:
       result = 16 - 3 - 4       result = 16 - 3
       print(result)              result = result + 4
                                  print(result)
     
     Execute A → prints 9 ✓     Execute B → prints 17 ✗
  
  3. Score: A gets reward 1.0, B gets reward 0.0
  4. GRPO: reinforce A, discourage B
  
No reward model needed. Code execution IS the reward.
"""

# Let's build this. First, a safe code executor.
import subprocess
import sys

def execute_code(code, timeout=5):
    """Run Python code in a subprocess, return the output."""
    try:
        result = subprocess.run(
            [sys.executable, "-c", code],
            capture_output=True, text=True, timeout=timeout
        )
        return result.stdout.strip()
    except (subprocess.TimeoutExpired, Exception):
        return None

# Test it
print(execute_code("print(16 - 3 - 4)"))       # → 9
print(execute_code("print(48 / 2 + 24)"))       # → 48.0
print(execute_code("while True: pass"))          # → None (timeout)
print(execute_code("import os; os.system('rm')"))  # → safe, subprocess is isolated


9
48.0
None



In [10]:
# The reward function: execute code, check if answer matches
def compute_reward(generated_code, correct_answer):
    """
    Execute the generated code, compare output to correct answer.
    Returns:
        1.0 = correct
        0.1 = code ran but wrong answer
        0.0 = code crashed or timed out
    """
    output = execute_code(generated_code)
    
    if output is None:
        return 0.0  # crashed or timed out
    
    try:
        # Compare numerically (handles "9" vs "9.0")
        if abs(float(output) - float(correct_answer)) < 0.01:
            return 1.0  # correct!
        else:
            return 0.1  # ran but wrong — still better than crashing
    except ValueError:
        return 0.0  # output wasn't a number

# Test it
print("Correct code:")
print(f"  reward = {compute_reward('print(16 - 3 - 4)', '9')}")

print("\nWrong answer:")
print(f"  reward = {compute_reward('print(16 + 3 + 4)', '9')}")

print("\nCrashing code:")
print(f"  reward = {compute_reward('print(x)', '9')}")

print("\nInfinite loop:")
print(f"  reward = {compute_reward('while True: pass', '9')}")


Correct code:
  reward = 1.0

Wrong answer:
  reward = 0.1

Crashing code:
  reward = 0.0

Infinite loop:
  reward = 0.0


In [11]:
from torch.nn.functional import log_softmax

# Fresh model for GRPO
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

grpo_model = AutoModelForCausalLM.from_pretrained("gpt2").to("cuda")
grpo_tokenizer = AutoTokenizer.from_pretrained("gpt2")
grpo_tokenizer.pad_token = grpo_tokenizer.eos_token

# Freeze base, add LoRA
for p in grpo_model.parameters():
    p.requires_grad = False
grpo_model = get_peft_model(grpo_model, LoraConfig(
    r=8, lora_alpha=16, target_modules=["c_attn", "c_proj"], lora_dropout=0.1,
))

# Simple math problems with answers
problems = [
    {"question": "What is 5 + 3?", "answer": "8"},
    {"question": "What is 10 - 4?", "answer": "6"},
    {"question": "What is 7 * 2?", "answer": "14"},
    {"question": "What is 20 / 4?", "answer": "5.0"},
    {"question": "What is 15 - 7?", "answer": "8"},
    {"question": "What is 3 * 9?", "answer": "27"},
    {"question": "What is 100 / 10?", "answer": "10.0"},
    {"question": "What is 6 + 9?", "answer": "15"},
]

PROMPT_TEMPLATE = """Write Python code that prints the answer.
Question: {question}
Code:
print("""

print("Model loaded. Ready for GRPO training.")
print(f"Trainable params: {sum(p.numel() for p in grpo_model.parameters() if p.requires_grad):,}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model loaded. Ready for GRPO training.
Trainable params: 811,008


/home/manas/anaconda3/envs/math-reasoning-distill/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [13]:
import random

In [14]:
import plotly.graph_objects as go
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter("runs/grpo_gpt2")
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, grpo_model.parameters()), lr=1e-4
)

N_SAMPLES = 4       # generate 4 solutions per problem
N_STEPS = 80        # training steps
MAX_NEW_TOKENS = 20 # max tokens for code generation

step_rewards = []
step_losses = []

grpo_model.train()
print("GRPO Training — GPT-2 learning to write math code\n")

for step in range(N_STEPS):
    # Pick a random problem
    prob = random.choice(problems)
    prompt = PROMPT_TEMPLATE.format(question=prob["question"])
    prompt_ids = grpo_tokenizer.encode(prompt, return_tensors="pt").to("cuda")
    prompt_len = prompt_ids.shape[1]

    # Step 1: Generate N solutions
    solutions = []
    with torch.no_grad():
        for _ in range(N_SAMPLES):
            output_ids = grpo_model.generate(
                prompt_ids, max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True, temperature=0.8, top_k=50,
                pad_token_id=grpo_tokenizer.eos_token_id,
            )
            generated = grpo_tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True)
            # Extract just the first line (the print statement)
            code_line = "print(" + generated.split(")")[0] + ")" if ")" in generated else "print(0)"
            solutions.append({"ids": output_ids, "code": code_line})

    # Step 2: Score each solution by executing the code
    rewards = []
    for sol in solutions:
        r = compute_reward(sol["code"], prob["answer"])
        rewards.append(r)

    # Step 3: Compute advantages
    mean_reward = sum(rewards) / len(rewards)
    advantages = [r - mean_reward for r in rewards]
    avg_reward = mean_reward

    # Step 4: GRPO policy update
    # For each solution, compute log probability and weight by advantage
    loss = torch.tensor(0.0, device="cuda")
    for sol, adv in zip(solutions, advantages):
        if abs(adv) < 1e-6:
            continue  # skip neutral solutions
        
        ids = sol["ids"].to("cuda")
        outputs = grpo_model(ids, labels=ids)
        # Weight the loss by advantage:
        #   positive advantage → reduce loss (reinforce)
        #   negative advantage → increase loss (discourage)
        loss += -adv * outputs.loss

    if loss.requires_grad:
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    step_rewards.append(avg_reward)
    step_losses.append(loss.item() if isinstance(loss, torch.Tensor) else 0)
    writer.add_scalar("reward/step", avg_reward, step)
    writer.add_scalar("loss/step", loss.item() if isinstance(loss, torch.Tensor) else 0, step)

    if step % 10 == 0:
        # Show what the model generated this step
        codes = [s["code"][:40] for s in solutions]
        print(f"Step {step:>3d} | Q: {prob['question']:<20s} | avg_reward: {avg_reward:.2f}")
        for i, (c, r) in enumerate(zip(codes, rewards)):
            print(f"         sol{i+1}: {c:<40s} reward={r:.1f}")
        print()

writer.close()
grpo_model.eval()
print("Done!\n")

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(len(step_rewards))), y=step_rewards,
    mode='lines', name='Avg Reward',
    line=dict(color='#00CC96', width=1),
))
# Moving average
window = 10
if len(step_rewards) > window:
    moving_avg = [sum(step_rewards[max(0,i-window):i+1])/len(step_rewards[max(0,i-window):i+1]) 
                  for i in range(len(step_rewards))]
    fig.add_trace(go.Scatter(
        x=list(range(len(moving_avg))), y=moving_avg,
        mode='lines', name=f'Moving Avg ({window})',
        line=dict(color='#EF553B', width=3),
    ))
fig.update_layout(
    title="GRPO Training — Reward Over Time (higher = more correct code)",
    xaxis_title="Step", yaxis_title="Avg Reward",
    template="plotly_dark", height=400, yaxis_range=[0, 1.1],
)
fig.show()


GRPO Training — GPT-2 learning to write math code

Step   0 | Q: What is 5 + 3?       | avg_reward: 0.05
         sol1: print(1, 10)                             reward=0.0
         sol2: print(3)                                 reward=0.1
         sol3: print(5)                                 reward=0.1
         sol4: print(question)                          reward=0.0

Step  10 | Q: What is 7 * 2?       | avg_reward: 0.10
         sol1: print(7)                                 reward=0.1
         sol2: print(7)                                 reward=0.1
         sol3: print(7)                                 reward=0.1
         sol4: print(5 * 4 )                            reward=0.1

Step  20 | Q: What is 10 - 4?      | avg_reward: 0.05
         sol1: print(5(3)                               reward=0.0
         sol2: print(number=1:Number2)                  reward=0.0
         sol3: print(10)                                reward=0.1
         sol4: print(100)                       